# English → Amharic Neural Machine Translation — Complete Notebook

This single notebook runs the **entire project** from start to finish:

| Section | What it does |
|---|---|
| **Part 1** | Dataset download, cleaning, vocab building, splits |
| **Part 2** | Basic Seq2Seq + LSTM training |
| **Part 3** | Attention-Based Seq2Seq + LSTM training |
| **Part 4** | BLEU, chrF, comparison table |
| **Part 5** | Error analysis & attention heatmaps |

**Dataset:** `michsethowusu/english-amharic_sentence-pairs_mt560` (CC BY 4.0, 669 k pairs)  
**Run all cells top-to-bottom — no manual steps needed.**

## 0. Install dependencies

In [ ]:
%pip install -q datasets pandas numpy matplotlib seaborn scikit-learn sacrebleu nltk tqdm torch torchvision streamlit

## 0.1 Global imports

In [ ]:
import os, re, json, time, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

import sacrebleu

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
print("sacrebleu:", sacrebleu.__version__)

## 0.2 Reproducibility, device, paths

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

PROJECT_ROOT  = Path('.').resolve()
DATA_DIR      = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR     = PROJECT_ROOT / 'models'
RESULTS_DIR   = PROJECT_ROOT / 'results'
ATTENTION_DIR = RESULTS_DIR  / 'attention'

for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR, ATTENTION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

---
# PART 1 — Dataset & Preprocessing
---

### 1.1 Download dataset

In [ ]:
dataset = load_dataset('michsethowusu/english-amharic_sentence-pairs_mt560')
df_raw  = dataset['train'].to_pandas()
print('Raw shape:', df_raw.shape)
df_raw.head(3)

### 1.2 Tokenisers

In [ ]:
def tokenize_english(text):
    return re.findall(r'\w+|[^\w\s]', str(text).strip().lower(), re.UNICODE)

def tokenize_amharic(text):
    return re.findall(r'\w+|[^\w\s]', str(text).strip(), re.UNICODE)

print(tokenize_english('I am going to the university.'))
print(tokenize_amharic('እኔ ወደ ዩኒቨርሲቲ እሄዳለሁ።'))

### 1.3 Clean & normalise

In [ ]:
def clean_english(text):
    text = str(text).strip()
    text = re.sub(r' +', ' ', text)
    return re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text)

def clean_amharic(text):
    text = str(text).strip()
    text = re.sub(r'[፡]', ' ', text)
    text = re.sub(r' +', ' ', text)
    return re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text).strip()

df = df_raw.copy()
df['eng'] = df['eng'].apply(clean_english)
df['amh'] = df['amh'].apply(clean_amharic)
print('After cleaning:', df.shape)

### 1.4 Remove missing / duplicates

In [ ]:
before = len(df)
df.dropna(subset=['eng','amh'], inplace=True)
df = df[df['eng'].str.strip().str.len() > 0]
df = df[df['amh'].str.strip().str.len() > 0]
df.drop_duplicates(subset=['eng','amh'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Removed {before - len(df)} rows. Remaining: {len(df):,}')

### 1.5 Length analysis & filtering

In [ ]:
df['eng_words'] = df['eng'].apply(lambda x: len(tokenize_english(x)))
df['amh_words'] = df['amh'].apply(lambda x: len(tokenize_amharic(x)))
print(df[['eng_words','amh_words']].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['eng_words'].clip(upper=80), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('English sentence lengths'); axes[0].set_xlabel('Tokens')
axes[1].hist(df['amh_words'].clip(upper=80), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Amharic sentence lengths'); axes[1].set_xlabel('Tokens')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'sentence_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

before = len(df)
df = df[(df['eng_words'].between(1,80)) & (df['amh_words'].between(1,80))].reset_index(drop=True)
print(f'After length filter: {len(df):,} pairs')

### 1.6 Train / Validation / Test split (80 / 10 / 10)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)
for d in [train_df, val_df, test_df]: d.reset_index(drop=True, inplace=True)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

train_df.to_csv(DATA_DIR/'train.csv',      index=False, encoding='utf-8-sig')
val_df.to_csv(  DATA_DIR/'validation.csv', index=False, encoding='utf-8-sig')
test_df.to_csv( DATA_DIR/'test.csv',       index=False, encoding='utf-8-sig')
print('Splits saved.')

### 1.7 Build & save vocabularies

In [ ]:
PAD_TOKEN = '<pad>'; UNK_TOKEN = '<unk>'; SOS_TOKEN = '<sos>'; EOS_TOKEN = '<eos>'
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

def build_vocab(sentences, tokenizer, min_freq=2):
    counter = Counter()
    for s in sentences: counter.update(tokenizer(s))
    vocab = {t: i for i, t in enumerate(SPECIAL_TOKENS)}
    for w, f in counter.most_common():
        if f >= min_freq and w not in vocab:
            vocab[w] = len(vocab)
    return vocab

print('Building English vocab...')
en_vocab  = build_vocab(train_df['eng'].tolist(), tokenize_english, min_freq=2)
print('Building Amharic vocab...')
amh_vocab = build_vocab(train_df['amh'].tolist(), tokenize_amharic, min_freq=2)

with open(MODEL_DIR/'en_vocab.json',  'w', encoding='utf-8') as f: json.dump(en_vocab,  f, ensure_ascii=False, indent=2)
with open(MODEL_DIR/'amh_vocab.json', 'w', encoding='utf-8') as f: json.dump(amh_vocab, f, ensure_ascii=False, indent=2)

print(f'English vocab : {len(en_vocab):,}')
print(f'Amharic vocab : {len(amh_vocab):,}')
print('Vocabularies saved.')

### 1.8 Top-40 word frequency chart

In [ ]:
en_c = Counter(); amh_c = Counter()
for s in train_df['eng']: en_c.update(tokenize_english(s))
for s in train_df['amh']: amh_c.update(tokenize_amharic(s))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
en_top  = en_c.most_common(40);  amh_top = amh_c.most_common(40)
axes[0].bar([w for w,_ in en_top],  [c for _,c in en_top],  color='steelblue')
axes[0].set_title('Top 40 English words'); axes[0].tick_params(axis='x', rotation=90)
axes[1].bar([w for w,_ in amh_top], [c for _,c in amh_top], color='darkorange')
axes[1].set_title('Top 40 Amharic words'); axes[1].tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(RESULTS_DIR/'top_words.png', dpi=150, bbox_inches='tight'); plt.show()
print('Part 1 complete.')

---
# PART 2 — Basic Seq2Seq + LSTM
---

### 2.1 Special token indices & helpers

In [ ]:
EN_PAD_IDX  = en_vocab[PAD_TOKEN];  EN_UNK_IDX  = en_vocab[UNK_TOKEN]
EN_SOS_IDX  = en_vocab[SOS_TOKEN];  EN_EOS_IDX  = en_vocab[EOS_TOKEN]
AMH_PAD_IDX = amh_vocab[PAD_TOKEN]; AMH_UNK_IDX = amh_vocab[UNK_TOKEN]
AMH_SOS_IDX = amh_vocab[SOS_TOKEN]; AMH_EOS_IDX = amh_vocab[EOS_TOKEN]
MAX_LEN = 40

en_itos  = {int(v): k for k, v in en_vocab.items()}
amh_itos = {int(v): k for k, v in amh_vocab.items()}

def numericalize(tokens, vocab, unk_idx):
    return [vocab.get(t, unk_idx) for t in tokens]

def decode_ids(ids, itos):
    tokens = []
    for i in ids:
        tok = itos.get(int(i), UNK_TOKEN)
        if tok == EOS_TOKEN: break
        if tok in (SOS_TOKEN, PAD_TOKEN): continue
        tokens.append(tok)
    return re.sub(r'\s+([።፣፤,.!?;:])', r'\1', ' '.join(tokens))

print('EN  — PAD:', EN_PAD_IDX, ' SOS:', EN_SOS_IDX)
print('AMH — PAD:', AMH_PAD_IDX,' SOS:', AMH_SOS_IDX)

### 2.2 Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.src = dataframe['eng'].astype(str).tolist()
        self.trg = dataframe['amh'].astype(str).tolist()
    def __len__(self): return len(self.src)
    def __getitem__(self, idx):
        src_ids = [EN_SOS_IDX]  + numericalize(tokenize_english(self.src[idx])[:MAX_LEN], en_vocab,  EN_UNK_IDX)  + [EN_EOS_IDX]
        trg_ids = [AMH_SOS_IDX] + numericalize(tokenize_amharic(self.trg[idx])[:MAX_LEN], amh_vocab, AMH_UNK_IDX) + [AMH_EOS_IDX]
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(trg_ids, dtype=torch.long)

def collate_fn(batch):
    srcs, trgs = zip(*batch)
    return pad_sequence(srcs, padding_value=EN_PAD_IDX), pad_sequence(trgs, padding_value=AMH_PAD_IDX)

BATCH_SIZE = 64
train_dataset = TranslationDataset(train_df)
val_dataset   = TranslationDataset(val_df)
test_dataset  = TranslationDataset(test_df)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader  = DataLoader(test_dataset,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
print('Train batches:', len(train_loader))

### 2.3 Encoder / Decoder / Seq2Seq model

In [ ]:
class S2S_Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=EN_PAD_IDX)
        self.rnn       = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers,
                                 dropout=dropout if n_layers>1 else 0.)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, src):
        _, (h, c) = self.rnn(self.dropout(self.embedding(src)))
        return h, c

class S2S_Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding  = nn.Embedding(output_dim, emb_dim, padding_idx=AMH_PAD_IDX)
        self.rnn        = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers,
                                  dropout=dropout if n_layers>1 else 0.)
        self.fc_out     = nn.Linear(hid_dim, output_dim)
        self.dropout    = nn.Dropout(dropout)
    def forward(self, token, h, c):
        emb = self.dropout(self.embedding(token.unsqueeze(0)))
        out, (h, c) = self.rnn(emb, (h, c))
        return self.fc_out(out.squeeze(0)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder; self.decoder = decoder; self.device = device
    def forward(self, src, trg, tf_ratio=0.5):
        outputs = torch.zeros(trg.shape[0], trg.shape[1], self.decoder.output_dim, device=self.device)
        h, c = self.encoder(src)
        tok  = trg[0]
        for t in range(1, trg.shape[0]):
            out, h, c = self.decoder(tok, h, c)
            outputs[t] = out
            tok = trg[t] if random.random() < tf_ratio else out.argmax(1)
        return outputs

INPUT_DIM  = len(en_vocab)
OUTPUT_DIM = len(amh_vocab)

s2s_model = Seq2Seq(
    S2S_Encoder(INPUT_DIM,  256, 512, 1, 0.2),
    S2S_Decoder(OUTPUT_DIM, 256, 512, 1, 0.2),
    device
).to(device)

def init_weights(m):
    for n, p in m.named_parameters():
        nn.init.xavier_uniform_(p) if p.dim()>=2 else nn.init.constant_(p, 0.)
s2s_model.apply(init_weights)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
s2s_params = count_params(s2s_model)
print(f'Seq2Seq parameters: {s2s_params:,}')

### 2.4 Training helpers

In [ ]:
LEARNING_RATE = 0.001; GRAD_CLIP = 1.0; TF_RATIO = 0.5; N_EPOCHS = 15

criterion = nn.CrossEntropyLoss(ignore_index=AMH_PAD_IDX)

def train_epoch(model, loader, opt, criterion, clip, tf):
    model.train(); loss_sum = 0.
    for src, trg in tqdm(loader, desc='  train', leave=False):
        src, trg = src.to(device), trg.to(device)
        opt.zero_grad(set_to_none=True)
        out = model(src, trg, tf)
        loss = criterion(out[1:].reshape(-1, out.shape[-1]), trg[1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step(); loss_sum += loss.item()
    return loss_sum / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, criterion, is_attn=False):
    model.eval(); loss_sum = 0.
    for src, trg in tqdm(loader, desc='  eval ', leave=False):
        src, trg = src.to(device), trg.to(device)
        out = model(src, trg, 0.) if not is_attn else model(src, trg, 0.)[0]
        loss = criterion(out[1:].reshape(-1, out.shape[-1]), trg[1:].reshape(-1))
        loss_sum += loss.item()
    return loss_sum / len(loader)

### 2.5 Train Seq2Seq

In [ ]:
s2s_opt = torch.optim.Adam(s2s_model.parameters(), lr=LEARNING_RATE)
best_val = float('inf'); s2s_train_losses = []; s2s_val_losses = []
s2s_best_path = MODEL_DIR / 'seq2seq_lstm_best.pt'

t_start = time.perf_counter()
for epoch in range(1, N_EPOCHS+1):
    t0 = time.perf_counter()
    tl = train_epoch(s2s_model, train_loader, s2s_opt, criterion, GRAD_CLIP, TF_RATIO)
    vl = eval_epoch(s2s_model,  val_loader,   criterion)
    s2s_train_losses.append(tl); s2s_val_losses.append(vl)
    mark = ''
    if vl < best_val:
        best_val = vl; torch.save(s2s_model.state_dict(), s2s_best_path); mark = '  ✓'
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | Train {tl:.4f} | Val {vl:.4f} | {(time.perf_counter()-t0)/60:.2f} min{mark}')

s2s_training_time = time.perf_counter() - t_start
print(f'\nTotal: {s2s_training_time/60:.2f} min  Best val loss: {best_val:.4f}')

### 2.6 Training curves

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(s2s_train_losses, marker='o', label='Train')
plt.plot(s2s_val_losses,   marker='o', label='Val')
plt.title('Basic Seq2Seq LSTM — Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(RESULTS_DIR/'seq2seq_training_curves.png', dpi=150); plt.show()

### 2.7 Load best checkpoint & greedy inference

In [ ]:
s2s_model.load_state_dict(torch.load(s2s_best_path, map_location=device))
s2s_model.eval()

@torch.no_grad()
def translate_s2s(sentence, max_len=MAX_LEN):
    src_ids = [EN_SOS_IDX] + [en_vocab.get(t, EN_UNK_IDX) for t in tokenize_english(sentence)[:max_len]] + [EN_EOS_IDX]
    src_t   = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(1)
    h, c    = s2s_model.encoder(src_t)
    tok = torch.tensor([AMH_SOS_IDX], dtype=torch.long, device=device)
    gen = []
    for _ in range(max_len):
        out, h, c = s2s_model.decoder(tok, h, c)
        p = out.argmax(1).item()
        if p == AMH_EOS_IDX: break
        gen.append(p)
        tok = torch.tensor([p], dtype=torch.long, device=device)
    return decode_ids(gen, amh_itos)

for s in ['I am a student.', 'I am going to the university.', 'God created the heavens and the earth.']:
    print(f'EN : {s}\nAMH: {translate_s2s(s)}\n')

### 2.8 Seq2Seq test loss, timing & save

In [ ]:
s2s_test_loss = eval_epoch(s2s_model, test_loader, criterion)
print(f'Seq2Seq test loss: {s2s_test_loss:.4f}')

timing_sents = test_df['eng'].astype(str).head(100).tolist()
for s in timing_sents[:5]: translate_s2s(s)   # warm-up
t0 = time.perf_counter()
for s in timing_sents: translate_s2s(s)
s2s_inf_ms = (time.perf_counter()-t0)/len(timing_sents)*1000
print(f'Avg inference: {s2s_inf_ms:.2f} ms')

torch.save(s2s_model.state_dict(), MODEL_DIR/'seq2seq_lstm.pt')

s2s_cfg = dict(model='Basic Seq2Seq + LSTM', embedding_dim=256, hidden_dim=512,
               encoder_layers=1, decoder_layers=1, batch_size=BATCH_SIZE,
               learning_rate=LEARNING_RATE, optimizer='Adam', epochs=N_EPOCHS,
               parameter_count=s2s_params, test_loss=s2s_test_loss,
               training_time_seconds=s2s_training_time, avg_inference_ms=s2s_inf_ms)
with open(MODEL_DIR/'seq2seq_lstm_config.json','w',encoding='utf-8') as f:
    json.dump(s2s_cfg, f, indent=2)
print('Seq2Seq model & config saved.')

---
# PART 3 — Attention-Based Seq2Seq + LSTM (Bahdanau)
---

### 3.1 Attention model architecture

In [ ]:
class AttnEncoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid, dec_hid, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=EN_PAD_IDX)
        self.rnn       = nn.LSTM(emb_dim, enc_hid, num_layers=n_layers,
                                 dropout=dropout if n_layers>1 else 0.)
        self.fc_h      = nn.Linear(enc_hid, dec_hid)
        self.fc_c      = nn.Linear(enc_hid, dec_hid)
        self.dropout   = nn.Dropout(dropout)
    def forward(self, src):
        out, (h, c) = self.rnn(self.dropout(self.embedding(src)))
        return out, torch.tanh(self.fc_h(h)), torch.tanh(self.fc_c(c))

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hid, dec_hid, attn_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hid + dec_hid, attn_dim)
        self.v    = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, dec_h, enc_out, mask):
        src_len = enc_out.shape[0]
        enc = enc_out.permute(1,0,2)
        dec = dec_h.unsqueeze(1).repeat(1, src_len, 1)
        e   = torch.tanh(self.attn(torch.cat((dec, enc), dim=2)))
        a   = self.v(e).squeeze(2).masked_fill(mask==0, -1e10)
        return torch.softmax(a, dim=1)

class AttnDecoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid, dec_hid, attn_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding  = nn.Embedding(output_dim, emb_dim, padding_idx=AMH_PAD_IDX)
        self.attention  = BahdanauAttention(enc_hid, dec_hid, attn_dim)
        self.rnn        = nn.LSTM(emb_dim+enc_hid, dec_hid, num_layers=n_layers,
                                  dropout=dropout if n_layers>1 else 0.)
        self.fc_out     = nn.Linear(dec_hid+enc_hid+emb_dim, output_dim)
        self.dropout    = nn.Dropout(dropout)
    def forward(self, tok, h, c, enc_out, mask):
        emb = self.dropout(self.embedding(tok.unsqueeze(0)))
        aw  = self.attention(h[-1], enc_out, mask)
        ctx = torch.bmm(aw.unsqueeze(1), enc_out.permute(1,0,2)).permute(1,0,2)
        out, (h, c) = self.rnn(torch.cat((emb, ctx), dim=2), (h, c))
        pred = self.fc_out(torch.cat((out, ctx, emb), dim=2).squeeze(0))
        return pred, h, c, aw

class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder; self.decoder = decoder; self.device = device
    def create_mask(self, src): return (src != EN_PAD_IDX).permute(1, 0)
    def forward(self, src, trg, tf_ratio=0.5):
        outputs    = torch.zeros(trg.shape[0], trg.shape[1], self.decoder.output_dim, device=self.device)
        attentions = torch.zeros(trg.shape[0]-1, trg.shape[1], src.shape[0], device=self.device)
        enc_out, h, c = self.encoder(src)
        mask = self.create_mask(src); tok = trg[0]
        for t in range(1, trg.shape[0]):
            out, h, c, aw = self.decoder(tok, h, c, enc_out, mask)
            outputs[t] = out; attentions[t-1] = aw
            tok = trg[t] if random.random() < tf_ratio else out.argmax(1)
        return outputs, attentions

attn_model = AttentionSeq2Seq(
    AttnEncoder(INPUT_DIM,  256, 512, 512, 1, 0.2),
    AttnDecoder(OUTPUT_DIM, 256, 512, 512, 256, 1, 0.2),
    device
).to(device)
attn_model.apply(init_weights)
attn_params = count_params(attn_model)
print(f'Attention model parameters: {attn_params:,}')

### 3.2 Train Attention model

In [ ]:
attn_opt  = torch.optim.Adam(attn_model.parameters(), lr=LEARNING_RATE)
best_val2 = float('inf'); attn_train_losses = []; attn_val_losses = []
attn_best_path = MODEL_DIR / 'attention_lstm_best.pt'

t_start2 = time.perf_counter()
for epoch in range(1, N_EPOCHS+1):
    t0 = time.perf_counter()
    # train
    attn_model.train(); tl = 0.
    for src, trg in tqdm(train_loader, desc='  train', leave=False):
        src, trg = src.to(device), trg.to(device)
        attn_opt.zero_grad(set_to_none=True)
        out, _ = attn_model(src, trg, TF_RATIO)
        loss = criterion(out[1:].reshape(-1, out.shape[-1]), trg[1:].reshape(-1))
        loss.backward(); torch.nn.utils.clip_grad_norm_(attn_model.parameters(), GRAD_CLIP)
        attn_opt.step(); tl += loss.item()
    tl /= len(train_loader)
    vl = eval_epoch(attn_model, val_loader, criterion, is_attn=True)
    attn_train_losses.append(tl); attn_val_losses.append(vl)
    mark = ''
    if vl < best_val2:
        best_val2 = vl; torch.save(attn_model.state_dict(), attn_best_path); mark = '  ✓'
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | Train {tl:.4f} | Val {vl:.4f} | {(time.perf_counter()-t0)/60:.2f} min{mark}')

attn_training_time = time.perf_counter() - t_start2
print(f'\nTotal: {attn_training_time/60:.2f} min  Best val loss: {best_val2:.4f}')

### 3.3 Training curves

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(attn_train_losses, marker='o', label='Train')
plt.plot(attn_val_losses,   marker='o', label='Val')
plt.title('Attention-LSTM — Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(RESULTS_DIR/'attention_training_curves.png', dpi=150); plt.show()

### 3.4 Load best checkpoint & greedy inference with attention

In [ ]:
attn_model.load_state_dict(torch.load(attn_best_path, map_location=device))
attn_model.eval()

@torch.no_grad()
def translate_attn(sentence, max_len=MAX_LEN):
    src_tokens = tokenize_english(sentence)[:max_len]
    src_ids    = [EN_SOS_IDX] + [en_vocab.get(t, EN_UNK_IDX) for t in src_tokens] + [EN_EOS_IDX]
    src_t      = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(1)
    enc_out, h, c = attn_model.encoder(src_t)
    mask = attn_model.create_mask(src_t)
    tok  = torch.tensor([AMH_SOS_IDX], dtype=torch.long, device=device)
    gen, attn_rows = [], []
    for _ in range(max_len):
        out, h, c, aw = attn_model.decoder(tok, h, c, enc_out, mask)
        p = out.argmax(1).item()
        if p == AMH_EOS_IDX: break
        gen.append(p); attn_rows.append(aw.squeeze(0).cpu().numpy())
        tok = torch.tensor([p], dtype=torch.long, device=device)
    translation  = decode_ids(gen, amh_itos)
    attn_matrix  = np.array(attn_rows) if attn_rows else np.zeros((1, len(src_ids)))
    return translation, attn_matrix, src_tokens

for s in ['I am a student.', 'I am going to the university.', 'God created the heavens and the earth.']:
    t, _, _ = translate_attn(s)
    print(f'EN : {s}\nAMH: {t}\n')

### 3.5 Attention heatmaps (5 examples)

In [ ]:
def plot_attention(src_tokens, tgt_text, attn_matrix, title='Bahdanau Attention', save_path=None):
    tgt_tokens = tokenize_amharic(tgt_text) if tgt_text.strip() else ['(empty)']
    n_trg = min(len(tgt_tokens), attn_matrix.shape[0])
    n_src = min(len(src_tokens), attn_matrix.shape[1])
    if n_trg==0 or n_src==0: return
    m = attn_matrix[:n_trg, :n_src]
    m = m / np.maximum(m.sum(axis=1, keepdims=True), 1e-12)
    fig, ax = plt.subplots(figsize=(max(7, n_src*.65), max(4, n_trg*.55)))
    im = ax.imshow(m, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(n_src)); ax.set_xticklabels(src_tokens[:n_src], rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(n_trg)); ax.set_yticklabels(tgt_tokens[:n_trg], fontsize=9)
    ax.set_xlabel('Source (English) tokens'); ax.set_ylabel('Generated Amharic tokens')
    ax.set_title(title); fig.colorbar(im, ax=ax, label='Attention weight')
    plt.tight_layout()
    if save_path: fig.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.show(); plt.close(fig)

for i in range(min(5, len(test_df))):
    src = test_df.iloc[i]['eng']; ref = test_df.iloc[i]['amh']
    pred, am, st = translate_attn(src)
    print(f'Example {i+1}\n  Source    : {src}\n  Reference : {ref}\n  Prediction: {pred}')
    plot_attention(st, pred, am, f'Attention — Example {i+1}',
                   save_path=ATTENTION_DIR/f'attention_example_{i+1}.png')

### 3.6 Attention test loss, timing & save

In [ ]:
attn_test_loss = eval_epoch(attn_model, test_loader, criterion, is_attn=True)
print(f'Attention test loss: {attn_test_loss:.4f}')

for s in timing_sents[:5]: translate_attn(s)   # warm-up
t0 = time.perf_counter()
for s in timing_sents: translate_attn(s)
attn_inf_ms = (time.perf_counter()-t0)/len(timing_sents)*1000
print(f'Avg inference: {attn_inf_ms:.2f} ms')

torch.save(attn_model.state_dict(), MODEL_DIR/'attention_lstm.pt')

attn_cfg = dict(model='Attention-Based Seq2Seq + LSTM', attention='Bahdanau',
                embedding_dim=256, hidden_dim=512, attn_dim=256,
                encoder_layers=1, decoder_layers=1, batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE, optimizer='Adam', epochs=N_EPOCHS,
                parameter_count=attn_params, test_loss=attn_test_loss,
                training_time_seconds=attn_training_time, avg_inference_ms=attn_inf_ms)
with open(MODEL_DIR/'attention_lstm_config.json','w',encoding='utf-8') as f:
    json.dump(attn_cfg, f, indent=2)
print('Attention model & config saved.')

---
# PART 4 — Evaluation & Comparison
---

### 4.1 Corpus BLEU & chrF on the full test set

In [ ]:
refs      = test_df['amh'].astype(str).tolist()
s2s_hyps  = [translate_s2s(s)        for s in tqdm(test_df['eng'].astype(str), desc='Seq2Seq  ')]
attn_hyps = [translate_attn(s)[0]    for s in tqdm(test_df['eng'].astype(str), desc='Attention')]

s2s_bleu  = sacrebleu.corpus_bleu(s2s_hyps,  [refs])
s2s_chrf  = sacrebleu.corpus_chrf(s2s_hyps,  [refs])
attn_bleu = sacrebleu.corpus_bleu(attn_hyps, [refs])
attn_chrf = sacrebleu.corpus_chrf(attn_hyps, [refs])

print(f'Seq2Seq   — BLEU: {s2s_bleu.score:.2f}  chrF: {s2s_chrf.score:.2f}')
print(f'Attention — BLEU: {attn_bleu.score:.2f}  chrF: {attn_chrf.score:.2f}')

### 4.2 Comparison table

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Basic Seq2Seq + LSTM',
     'BLEU':  round(s2s_bleu.score, 2),
     'chrF':  round(s2s_chrf.score, 2),
     'Test Loss':            round(s2s_test_loss, 4),
     'Training Time (min)': round(s2s_training_time/60, 2),
     'Avg Inference (ms)':  round(s2s_inf_ms, 2),
     'Parameters':          s2s_params},
    {'Model': 'Attention-Based Seq2Seq + LSTM',
     'BLEU':  round(attn_bleu.score, 2),
     'chrF':  round(attn_chrf.score, 2),
     'Test Loss':            round(attn_test_loss, 4),
     'Training Time (min)': round(attn_training_time/60, 2),
     'Avg Inference (ms)':  round(attn_inf_ms, 2),
     'Parameters':          attn_params},
])
print('='*80); print('MODEL COMPARISON TABLE'); print('='*80)
print(comparison.to_string(index=False))
comparison.to_csv(RESULTS_DIR/'model_comparison.csv', index=False, encoding='utf-8-sig')
print('Saved: results/model_comparison.csv')

### 4.3 Bar chart

In [ ]:
metrics = ['BLEU','chrF','Test Loss','Avg Inference (ms)']
s2s_v   = [comparison.loc[0,m] for m in metrics]
atn_v   = [comparison.loc[1,m] for m in metrics]
x = np.arange(len(metrics)); w = 0.35
fig, ax = plt.subplots(figsize=(11,5))
ax.bar(x-w/2, s2s_v, width=w, label='Basic Seq2Seq',     color='steelblue')
ax.bar(x+w/2, atn_v, width=w, label='Attention Seq2Seq', color='darkorange')
for xi,(v1,v2) in zip(x,zip(s2s_v,atn_v)):
    ax.text(xi-w/2, v1+max(v1,v2)*.01, f'{v1:.2f}', ha='center', fontsize=8)
    ax.text(xi+w/2, v2+max(v1,v2)*.01, f'{v2:.2f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title('Model Comparison'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR/'model_comparison_chart.png', dpi=150); plt.show()

### 4.4 Combined translation examples  (Source → Reference → Seq2Seq → Attention)

In [ ]:
rows = []
for i in range(min(20, len(test_df))):
    src = test_df.loc[i,'eng']; ref = test_df.loc[i,'amh']
    rows.append({'Source': src, 'Reference': ref,
                 'Seq2Seq Output': translate_s2s(src),
                 'Attention-LSTM Output': translate_attn(src)[0]})
combined_df = pd.DataFrame(rows)
combined_df.to_csv(RESULTS_DIR/'combined_translation_examples.csv', index=False, encoding='utf-8-sig')
print('Saved: results/combined_translation_examples.csv')
combined_df

### 4.5 Save evaluation JSON

In [ ]:
eval_results = {
    'seq2seq':   {'bleu': s2s_bleu.score,  'chrf': s2s_chrf.score,  'test_loss': s2s_test_loss,
                  'parameters': s2s_params,  'training_time_min': round(s2s_training_time/60,2),  'avg_inference_ms': s2s_inf_ms},
    'attention': {'bleu': attn_bleu.score, 'chrf': attn_chrf.score, 'test_loss': attn_test_loss,
                  'parameters': attn_params, 'training_time_min': round(attn_training_time/60,2), 'avg_inference_ms': attn_inf_ms},
}
with open(RESULTS_DIR/'evaluation_results.json','w',encoding='utf-8') as f:
    json.dump(eval_results, f, indent=2)
print('Saved: results/evaluation_results.json')

---
# PART 5 — Error & Attention Analysis
---

### 5.1 Error detection helpers

In [ ]:
def has_repeats(text, threshold=2):
    tokens = re.findall(r'\w+', text, re.UNICODE)
    return any(all(tokens[i+j]==tokens[i] for j in range(1,threshold+1))
               for i in range(len(tokens)-threshold))

def length_ratio(hyp, ref):
    h = len(re.findall(r'\w+', hyp, re.UNICODE))
    r = len(re.findall(r'\w+', ref, re.UNICODE))
    return h / max(r, 1)

def unk_count(text): return text.lower().count('<unk>')

def analyse(hyps, refs, srcs):
    rows = []
    for hyp, ref, src in zip(hyps, refs, srcs):
        try: sb = sacrebleu.sentence_bleu(hyp, [ref]).score
        except: sb = 0.
        lr = length_ratio(hyp, ref)
        rows.append({'source': src, 'reference': ref, 'hypothesis': hyp,
                     'sentence_bleu': sb, 'length_ratio': lr,
                     'repeated_words': has_repeats(hyp),
                     'unk_count': unk_count(hyp),
                     'src_length': len(tokenize_english(src)),
                     'missing_words': lr < 0.75,
                     'extra_words':   lr > 1.35})
    return pd.DataFrame(rows)

print('Error helpers ready.')

### 5.2 Quantitative error analysis

In [ ]:
srcs = test_df['eng'].astype(str).tolist()[:len(s2s_hyps)]
s2s_err  = analyse(s2s_hyps,  refs[:len(s2s_hyps)],  srcs)
attn_err = analyse(attn_hyps, refs[:len(attn_hyps)], srcs)

def summary(df, name):
    print(f'\n{name}')
    print(f'  Avg sentence BLEU : {df["sentence_bleu"].mean():.2f}')
    print(f'  Repeated words    : {df["repeated_words"].sum()} / {len(df)}')
    print(f'  Sentences with UNK: {(df["unk_count"]>0).sum()} / {len(df)}')
    print(f'  Missing words     : {df["missing_words"].sum()} / {len(df)}')
    print(f'  Extra words       : {df["extra_words"].sum()} / {len(df)}')

summary(s2s_err,  'Basic Seq2Seq + LSTM')
summary(attn_err, 'Attention-LSTM')

### 5.3 Error category bar chart

In [ ]:
cats = ['Repeated words','Contains UNK','Missing words','Extra words']
s2s_c  = [s2s_err['repeated_words'].sum(),  (s2s_err['unk_count']>0).sum(),  s2s_err['missing_words'].sum(),  s2s_err['extra_words'].sum()]
atn_c  = [attn_err['repeated_words'].sum(), (attn_err['unk_count']>0).sum(), attn_err['missing_words'].sum(), attn_err['extra_words'].sum()]
x = np.arange(len(cats)); w = 0.35
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x-w/2, s2s_c, width=w, label='Basic Seq2Seq',     color='steelblue')
ax.bar(x+w/2, atn_c, width=w, label='Attention Seq2Seq', color='darkorange')
for xi,(v1,v2) in zip(x,zip(s2s_c,atn_c)):
    ax.text(xi-w/2, v1+.3, str(int(v1)), ha='center', fontsize=9)
    ax.text(xi+w/2, v2+.3, str(int(v2)), ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_title('Error Category Count'); ax.legend(); ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR/'error_category_comparison.png', dpi=150); plt.show()

### 5.4 BLEU vs sentence length

In [ ]:
bins   = [0,5,10,15,20,30,40,100]
labels = ['1-5','6-10','11-15','16-20','21-30','31-40','41+']
s2s_err['len_bin']  = pd.cut(s2s_err['src_length'],  bins=bins, labels=labels)
attn_err['len_bin'] = pd.cut(attn_err['src_length'], bins=bins, labels=labels)
s2s_bl  = s2s_err.groupby('len_bin',  observed=True)['sentence_bleu'].mean()
attn_bl = attn_err.groupby('len_bin', observed=True)['sentence_bleu'].mean()
plt.figure(figsize=(10,4))
plt.plot(s2s_bl.index.astype(str),  s2s_bl.values,  marker='o', label='Basic Seq2Seq',     color='steelblue')
plt.plot(attn_bl.index.astype(str), attn_bl.values, marker='o', label='Attention Seq2Seq', color='darkorange')
plt.xlabel('Source length (tokens)'); plt.ylabel('Avg sentence BLEU')
plt.title('Translation quality vs. sentence length')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(RESULTS_DIR/'bleu_vs_length.png', dpi=150); plt.show()

### 5.5 Best & worst translation examples

In [ ]:
for label, df_err in [('Basic Seq2Seq', s2s_err), ('Attention-LSTM', attn_err)]:
    print(f'\n{'='*65}\n{label} — 3 BEST translations\n{'='*65}')
    for _, r in df_err.nlargest(3,'sentence_bleu').iterrows():
        print(f'  Source    : {r["source"]}\n  Reference : {r["reference"]}\n  Hypothesis: {r["hypothesis"]}\n  BLEU: {r["sentence_bleu"]:.1f}\n')
    print(f'\n{'='*65}\n{label} — 3 WORST translations\n{'='*65}')
    for _, r in df_err.nsmallest(3,'sentence_bleu').iterrows():
        print(f'  Source    : {r["source"]}\n  Reference : {r["reference"]}\n  Hypothesis: {r["hypothesis"]}\n  BLEU: {r["sentence_bleu"]:.1f}\n')

### 5.6 Extra attention heatmaps (3 more)

In [ ]:
for i in range(5, min(8, len(test_df))):
    src = test_df.iloc[i]['eng']; ref = test_df.iloc[i]['amh']
    pred, am, st = translate_attn(src)
    print(f'\nExample {i+1}\n  Source    : {src}\n  Reference : {ref}\n  Prediction: {pred}')
    plot_attention(st, pred, am, f'Attention — Example {i+1}',
                   save_path=ATTENTION_DIR/f'attention_example_{i+1}.png')

### 5.7 Save error analysis summary

In [ ]:
error_table = pd.DataFrame([
    {'Error Type': 'Repeated words',                  'Seq2Seq Count': int(s2s_err['repeated_words'].sum()),  'Seq2Seq %': round(100*s2s_err['repeated_words'].mean(),1),  'Attention Count': int(attn_err['repeated_words'].sum()),  'Attention %': round(100*attn_err['repeated_words'].mean(),1)},
    {'Error Type': 'Contains UNK token',              'Seq2Seq Count': int((s2s_err['unk_count']>0).sum()),   'Seq2Seq %': round(100*(s2s_err['unk_count']>0).mean(),1),   'Attention Count': int((attn_err['unk_count']>0).sum()),   'Attention %': round(100*(attn_err['unk_count']>0).mean(),1)},
    {'Error Type': 'Missing words (len ratio < 0.75)','Seq2Seq Count': int(s2s_err['missing_words'].sum()),   'Seq2Seq %': round(100*s2s_err['missing_words'].mean(),1),   'Attention Count': int(attn_err['missing_words'].sum()),   'Attention %': round(100*attn_err['missing_words'].mean(),1)},
    {'Error Type': 'Extra words (len ratio > 1.35)',  'Seq2Seq Count': int(s2s_err['extra_words'].sum()),     'Seq2Seq %': round(100*s2s_err['extra_words'].mean(),1),     'Attention Count': int(attn_err['extra_words'].sum()),     'Attention %': round(100*attn_err['extra_words'].mean(),1)},
])
print(error_table.to_string(index=False))
error_table.to_csv(RESULTS_DIR/'error_analysis_summary.csv', index=False, encoding='utf-8-sig')
print('Saved: results/error_analysis_summary.csv')

---
# Final Summary
---

In [ ]:
print('='*70)
print('COMPLETE PROJECT SUMMARY')
print('='*70)
print(f'\nDataset   : MT560 English-Amharic ({len(df):,} pairs after cleaning)')
print(f'Train/Val/Test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}')
print(f'English vocab  : {len(en_vocab):,} tokens')
print(f'Amharic vocab  : {len(amh_vocab):,} tokens')
print()
print(f"{'Metric':<26} {'Seq2Seq':>10} {'Attention':>10} {'Better':>10}")
print('-'*58)
rows_sum = [
    ('BLEU ↑',              s2s_bleu.score,      attn_bleu.score,      True),
    ('chrF ↑',              s2s_chrf.score,      attn_chrf.score,      True),
    ('Test Loss ↓',         s2s_test_loss,        attn_test_loss,       False),
    ('Inference ms ↓',      s2s_inf_ms,           attn_inf_ms,          False),
    ('Parameters',          float(s2s_params),    float(attn_params),   None),
    ('Training min',        s2s_training_time/60, attn_training_time/60,None),
]
for label, v1, v2, hib in rows_sum:
    if hib is True:   w = 'Seq2Seq' if v1>v2 else 'Attention' if v2>v1 else 'Tie'
    elif hib is False:w = 'Seq2Seq' if v1<v2 else 'Attention' if v2<v1 else 'Tie'
    else:             w = '—'
    print(f'{label:<26} {v1:>10.2f} {v2:>10.2f} {w:>10}')
print()
print('All models, results, and visualisations saved.')
print('Run   streamlit run app.py   to launch the translation app.')